# PDF Scan v3 - Executable Lab Notebook

This notebook is a thin bridge into the executable Phase A-E lab scripts.

Use it to configure inputs, run the phases in order, and inspect the structured artifacts written under `pdf-scan/runs/<run_id>/`.


## Step 0 - Inputs

Edit the next cell, then run the notebook top to bottom.

Use `INPUT_MODE = "small_gold"` for the benchmark suite or `INPUT_MODE = "manual"` to provide a chapter title, chapter description, and PDFs from `PDF_DIR` / `PDFS`.


In [ ]:
import os
from argparse import Namespace
from pathlib import Path

from phase_a_lab import *  # noqa: F401,F403
from phase_b_lab import PhaseBOptions, run_phase_b
from phase_c_lab import PhaseCOptions, run_phase_c
from phase_d_lab import PhaseDOptions, run_phase_d, phase_d_capabilities
from phase_e_lab import PhaseEOptions, run_phase_e, phase_e_capabilities


In [ ]:
INPUT_MODE = "small_gold"  # or "manual"
PIPELINE_VERSION = "pdf_scan_v3_notebook"

FORCE_REBUILD_PHASE_A = False
FORCE_REBUILD_PHASE_B = True
FORCE_REBUILD_PHASE_C = True
FORCE_REBUILD_PHASE_D = True
FORCE_REBUILD_PHASE_E = True

SUITE_MANIFEST = "benchmark/small_gold/manifests/suite_manifest.json"
CHAPTER_INDEX = 0
DOC_LIMIT = None
INCLUDE_DOC_IDS = []
EXCLUDE_DOC_IDS = []

CHAPTER_TITLE = ""
CHAPTER_DESCRIPTION = ""
PDFS = []
PDF_DIR = ""
PDF_GLOB = "*.pdf"
PDF_RECURSIVE = False
MAX_PDFS = 30

GROBID_BASE_URL = (os.getenv("GROBID_URL") or os.getenv("GROBID_BASE_URL") or "").strip()
PLANNER_MODEL = (os.getenv("OPENAI_PDF_SCAN_PLANNER_MODEL") or os.getenv("OPENAI_PDF_SCAN_MODEL") or "gpt-5-mini").strip() or "gpt-5-mini"
PLANNER_REASONING_EFFORT = "low"
EMBED_MODEL = (os.getenv("OPENAI_PDF_SCAN_EMBED_MODEL") or "text-embedding-3-small").strip() or "text-embedding-3-small"
USE_OPENAI_PLANNER = True
USE_OPENAI_DENSE = True


In [ ]:
# Phase A - Resolve inputs and create the run scaffold

phase_a_args = Namespace(
    input_mode=INPUT_MODE,
    pipeline_version=PIPELINE_VERSION,
    force_rebuild=bool(FORCE_REBUILD_PHASE_A),
    runs_root="",
    suite_manifest=SUITE_MANIFEST,
    chapter_index=int(CHAPTER_INDEX),
    doc_limit=DOC_LIMIT,
    include_doc_id=list(INCLUDE_DOC_IDS or []),
    exclude_doc_id=list(EXCLUDE_DOC_IDS or []),
    chapter_title=str(CHAPTER_TITLE or ""),
    chapter_description=str(CHAPTER_DESCRIPTION or ""),
    pdf=list(PDFS or []),
    pdf_dir=str(PDF_DIR or ""),
    pdf_glob=str(PDF_GLOB or "*.pdf"),
    pdf_recursive=bool(PDF_RECURSIVE),
    max_pdfs=int(MAX_PDFS),
)
phase_a_result = run_phase_a(phase_a_args)
RUN_CONTEXT = phase_a_result["run_ctx"]
PDF_MANIFEST = phase_a_result["manifest_rows"]
PHASE_A_SUMMARY = phase_a_result["summary"]
phase_a_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))

print_section("Phase A - What Happened")
print_kv({
    "run_id": RUN_CONTEXT.run_id,
    "run_dir": RUN_CONTEXT.run_dir,
    "phase_a_summary_json": phase_a_rel(RUN_CONTEXT.artifacts.phase_a_summary_json),
    "phase_status": phase_a_result["assessment"].get("status"),
    "pdf_count": len(PDF_MANIFEST),
})
print_section("Phase A - PDF Manifest")
print_table(PDF_MANIFEST, columns=["doc_id", "label", "pages", "file_name"], max_rows=30, max_col_width=48)
print_section("Phase A - QC")
print_table(PHASE_A_SUMMARY.get("qc_rows") or [], columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)


In [ ]:
# Phase B - Parse PDFs into per-document bundles

phase_b_logger = setup_run_logger(RUN_CONTEXT)
phase_b_options = PhaseBOptions(
    force_rebuild=bool(FORCE_REBUILD_PHASE_B),
    doc_limit=DOC_LIMIT,
    include_doc_ids=list(INCLUDE_DOC_IDS or []),
    exclude_doc_ids=list(EXCLUDE_DOC_IDS or []),
    min_page_words=20,
    min_doc_chars=200,
    try_docling=True,
    docling_page_limit=400,
    docling_max_file_size_bytes=50 * 1024 * 1024,
    docling_do_ocr=False,
    docling_do_table_structure=False,
    docling_document_timeout_sec=180,
    docling_num_threads=4,
    docling_enable_chunking=True,
    docling_chunk_size=20,
    docling_chunk_max_pages=400,
    docling_chunk_num_threads=1,
    try_grobid=True,
    grobid_page_limit=400,
    grobid_base_url=GROBID_BASE_URL,
    grobid_process_path="/api/processFulltextDocument",
    grobid_timeout_sec=120,
    grobid_consolidate_header=0,
    grobid_consolidate_citations=0,
    grobid_include_raw_citations=0,
)
with stage_timer(RUN_CONTEXT, "phase_b"):
    phase_b_result = run_phase_b(RUN_CONTEXT, PDF_MANIFEST, phase_b_options, stable_hash_fn=stable_hash, log_event_fn=log_event, run_logger=phase_b_logger)
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_b", {}).update(phase_b_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_b_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))
phase_b_rows = phase_b_result["summary_rows"]
docling_success_count = sum(1 for row in phase_b_rows if str(row.get("docling_status") or "") == "success")
docling_chunk_selected_count = sum(1 for row in phase_b_rows if bool(row.get("docling_chunk_selected")))
grobid_success_count = sum(1 for row in phase_b_rows if str(row.get("grobid_status") or "") == "success")
fallback_activated_docs = sum(1 for row in phase_b_rows if bool(row.get("fallback_activated")))

print_section("Phase B - Parser Capabilities")
print_kv({
    "fitz_available": phase_b_result["capabilities"].get("fitz_available"),
    "pypdf_available": phase_b_result["capabilities"].get("pypdf_available"),
    "docling_available": phase_b_result["capabilities"].get("docling_available"),
    "grobid_status": phase_b_result["capabilities"].get("grobid", {}).get("status"),
    "selected_documents": phase_b_result["selected_count"],
    "docling_success_count": docling_success_count,
    "docling_chunk_selected_count": docling_chunk_selected_count,
    "grobid_success_count": grobid_success_count,
})
print_section("Phase B - What Happened")
print_kv({
    "phase_b_summary_json": phase_b_rel(phase_b_result["summary_path"]),
    "phase_b_assessment_json": phase_b_rel(phase_b_result["assessment_path"]),
    "parsed_document_bundles_jsonl": phase_b_rel(phase_b_result["index_path"]),
    "documents_processed": len(phase_b_rows),
    "fallback_activated_docs": fallback_activated_docs,
    "phase_status": phase_b_result["assessment"].get("status"),
    "quality_band": phase_b_result["assessment"].get("quality_band"),
})
print_section("Phase B - Document Summary")
print_table(phase_b_rows, columns=["doc_id", "file_name", "page_count", "outline_count", "pages_with_text_pct", "docling_status", "docling_mode", "docling_section_header_count", "fallback_activated"], max_rows=30, max_col_width=44)
print_section("Phase B - QC")
print_table(phase_b_result["qc_rows"], columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)


In [ ]:
# Phase C - Normalize documents into sections and passages

phase_c_logger = setup_run_logger(RUN_CONTEXT)
phase_c_options = PhaseCOptions(
    force_rebuild=bool(FORCE_REBUILD_PHASE_C),
    doc_limit=DOC_LIMIT,
    include_doc_ids=list(INCLUDE_DOC_IDS or []),
    exclude_doc_ids=list(EXCLUDE_DOC_IDS or []),
    prefer_outline=True,
    use_docling=True,
    use_grobid=True,
    use_heuristic_headings=True,
    use_heuristic_recovery=True,
    repair_titles_from_anchor_blocks=True,
    heuristic_heading_min_words=1,
    heuristic_heading_max_words=18,
    heuristic_heading_max_chars=160,
    repeated_heading_page_threshold=3,
    min_section_chars=120,
    min_section_words=20,
    min_section_coverage_pct_warn=70.0,
    long_doc_page_threshold=40,
    passage_target_words=180,
    passage_max_words=260,
    passage_min_words=70,
    synthesize_front_matter=True,
    synthesize_document_body=True,
    metadata_filter_enabled=True,
    micro_section_max_words=20,
    micro_section_max_title_words=3,
)
with stage_timer(RUN_CONTEXT, "phase_c"):
    phase_c_result = run_phase_c(RUN_CONTEXT, phase_c_options, stable_hash_fn=stable_hash, log_event_fn=log_event, run_logger=phase_c_logger)
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_c", {}).update(phase_c_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_c_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))
phase_c_rows = phase_c_result["summary_rows"]
print_section("Phase C - Normalization Capabilities")
print_kv({
    "selected_documents": phase_c_result["selected_count"],
    "documents_jsonl": phase_c_rel(phase_c_result["documents_path"]),
    "sections_jsonl": phase_c_rel(phase_c_result["sections_path"]),
    "passages_jsonl": phase_c_rel(phase_c_result["passages_path"]),
})
print_section("Phase C - What Happened")
print_kv({
    "phase_c_summary_json": phase_c_rel(phase_c_result["summary_path"]),
    "phase_c_assessment_json": phase_c_rel(phase_c_result["assessment_path"]),
    "documents_processed": len(phase_c_rows),
    "sections_written": len(phase_c_result["section_rows"]),
    "passages_written": len(phase_c_result["passage_rows"]),
    "phase_status": phase_c_result["assessment"].get("status"),
    "quality_band": phase_c_result["assessment"].get("quality_band"),
})
print_section("Phase C - Document Summary")
print_table(phase_c_rows, columns=["doc_id", "file_name", "page_count", "strategy", "accepted_heading_count", "section_count", "passage_count", "section_coverage_pct", "fallback_anchor_count"], max_rows=30, max_col_width=44)
print_section("Phase C - QC")
print_table(phase_c_result["qc_rows"], columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)


In [ ]:
# Phase D - Build the query plan and retrieval views

phase_d_logger = setup_run_logger(RUN_CONTEXT)
phase_d_options = PhaseDOptions(
    force_rebuild=bool(FORCE_REBUILD_PHASE_D),
    use_openai_planner=bool(USE_OPENAI_PLANNER),
    allow_heuristic_fallback=True,
    openai_model=PLANNER_MODEL,
    reasoning_effort=PLANNER_REASONING_EFFORT,
    temperature=0.0,
    max_completion_tokens=1400,
    must_term_limit=8,
    should_term_limit=14,
    exclusion_limit=8,
    subpoint_limit=6,
    drift_risk_limit=8,
    source_anchor_limit=24,
    subpoint_source_anchor_limit=3,
    max_summary_chars=480,
    max_subpoint_summary_chars=320,
    min_anchor_token_overlap=0.67,
)
with stage_timer(RUN_CONTEXT, "phase_d"):
    phase_d_result = run_phase_d(RUN_CONTEXT, chapter_title=phase_a_result["config"].chapter_title, chapter_spec_text=phase_a_result["config"].chapter_spec_text, options=phase_d_options, stable_hash_fn=stable_hash, log_event_fn=log_event, run_logger=phase_d_logger)
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_d", {}).update(phase_d_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_d_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))
phase_d_plan = phase_d_result["query_plan"]
print_section("Phase D - Planner Capabilities")
print_kv({
    "openai_available": phase_d_capabilities().get("openai_available"),
    "pydantic_available": phase_d_capabilities().get("pydantic_available"),
    "openai_api_key_present": phase_d_capabilities().get("openai_api_key_present"),
    "planner_model": phase_d_options.openai_model,
    "planner_mode": phase_d_result["planner_trace"].get("planner_mode"),
    "api_mode": phase_d_result["planner_trace"].get("api_mode"),
    "pricing_source_url": (phase_d_result["planner_trace"].get("cost") or {}).get("pricing_source_url"),
})
print_section("Phase D - What Happened")
print_kv({
    "run_id": RUN_CONTEXT.run_id,
    "query_plan_json": phase_d_rel(phase_d_result["query_plan_path"]),
    "query_views_json": phase_d_rel(phase_d_result["query_views_path"]),
    "planner_prompt_json": phase_d_rel(phase_d_result["planner_prompt_path"]),
    "planner_response_json": phase_d_rel(phase_d_result["planner_response_path"]),
    "source_inventory_json": phase_d_rel(phase_d_result["source_inventory_path"]),
    "corpus_support_json": phase_d_rel(phase_d_result["corpus_support_path"]),
    "openai_input_tokens": (phase_d_result["planner_trace"].get("usage") or {}).get("input_tokens"),
    "openai_output_tokens": (phase_d_result["planner_trace"].get("usage") or {}).get("output_tokens"),
    "openai_estimated_cost_usd": (phase_d_result["planner_trace"].get("cost") or {}).get("estimated_cost_usd"),
    "phase_status": phase_d_result["assessment"].get("status"),
})
print_section("Phase D - Query Plan Summary")
print_kv({
    "chapter_title": truncate_text(phase_d_plan.get("chapter_title"), max_len=110),
    "chapter_summary": truncate_text(phase_d_plan.get("chapter_summary"), max_len=220),
    "source_anchors": ", ".join((phase_d_plan.get("source_anchors") or [])[:10]),
    "must_terms": ", ".join(phase_d_plan.get("must_terms") or []),
    "should_terms": ", ".join((phase_d_plan.get("should_terms") or [])[:10]),
    "drift_risks": ", ".join(phase_d_plan.get("drift_risks") or []),
})
print_section("Phase D - Subpoints")
print_table(phase_d_result["subpoint_rows"], columns=["subpoint_id", "label", "summary", "source_anchors", "must_terms", "preferred_section_types"], max_rows=20, max_col_width=48)
print_section("Phase D - Retrieval Views")
print_table(phase_d_result["retrieval_view_rows"], columns=["view_id", "kind", "target_units", "anchor_terms", "query_word_count", "query_text"], max_rows=30, max_col_width=52)
print_section("Phase D - Corpus Support")
print_table(phase_d_result["term_support_rows"], columns=["kind", "term", "doc_hits", "section_hits", "title_hits", "text_hits", "example_titles"], max_rows=24, max_col_width=42)
print_section("Phase D - QC")
print_table(phase_d_result["qc_rows"], columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)


In [ ]:
# Phase E - Generate the high-recall section candidate pool

phase_e_logger = setup_run_logger(RUN_CONTEXT)
phase_e_options = PhaseEOptions(
    force_rebuild=bool(FORCE_REBUILD_PHASE_E),
    candidate_limit_per_lane=80,
    fused_candidate_limit=120,
    per_view_limit_multiplier=2,
    rrf_k=60,
    lexical_k1=1.2,
    lexical_b=0.75,
    use_openai_dense=bool(USE_OPENAI_DENSE),
    allow_lexical_only_fallback=True,
    openai_embedding_model=EMBED_MODEL,
    openai_timeout_sec=300,
    dense_batch_size=64,
    dense_section_max_chars=4200,
    dense_passage_max_chars=2400,
    dense_query_max_chars=1600,
    dense_dimensions=None,
    dense_min_similarity=0.05,
    top_candidate_preview_count=20,
    selection_strategy="xquad",
    use_supported_subpoint_selection=True,
    abstain_when_no_supported_subpoints=True,
    generic_evidence_bonus=0.01,
    generic_anchor_score_threshold=1.0,
    subpoint_min_supported_candidates=1,
    subpoint_max_preview_rows=10,
    diversity_lambda=0.45,
)
with stage_timer(RUN_CONTEXT, "phase_e"):
    phase_e_result = run_phase_e(RUN_CONTEXT, options=phase_e_options, stable_hash_fn=stable_hash, log_event_fn=log_event, run_logger=phase_e_logger)
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_e", {}).update(phase_e_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_e_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))
PHASE_E_RESULT = phase_e_result
print_section("Phase E - Retrieval Capabilities")
print_kv({
    "numpy_available": phase_e_capabilities().get("numpy_available"),
    "openai_available": phase_e_capabilities().get("openai_available"),
    "openai_api_key_present": phase_e_capabilities().get("openai_api_key_present"),
    "embedding_model": phase_e_options.openai_embedding_model,
    "dense_mode": phase_e_result["dense_trace"].get("dense_mode"),
    "pricing_source_url": (phase_e_result["dense_trace"].get("cost") or {}).get("pricing_source_url"),
    "pricing_verified_date": (phase_e_result["dense_trace"].get("cost") or {}).get("pricing_verified_date"),
})
print_section("Phase E - What Happened")
print_kv({
    "fused_candidates_jsonl": phase_e_rel(phase_e_result["fused_candidates_path"]),
    "phase_e_config_json": phase_e_rel(phase_e_result["config_path"]),
    "phase_e_runtime_json": phase_e_rel(phase_e_result["runtime_path"]),
    "phase_e_summary_json": phase_e_rel(phase_e_result["summary_path"]),
    "phase_e_assessment_json": phase_e_rel(phase_e_result["assessment_path"]),
    "phase_e_dense_trace_json": phase_e_rel(phase_e_result["dense_trace_path"]),
    "phase_e_subpoint_support_json": phase_e_rel(phase_e_result["subpoint_support_path"]),
    "lane_files": len(phase_e_result["lane_paths"]),
    "fused_candidates": len(phase_e_result["fused_candidate_rows"]),
    "embedding_input_tokens": (phase_e_result["dense_trace"].get("usage") or {}).get("input_tokens"),
    "embedding_total_tokens": (phase_e_result["dense_trace"].get("usage") or {}).get("total_tokens"),
    "embedding_estimated_cost_usd": (phase_e_result["dense_trace"].get("cost") or {}).get("estimated_cost_usd"),
    "phase_status": phase_e_result["assessment"].get("status"),
})
print_section("Phase E - Subpoint Support")
print_kv({
    "selection_strategy": phase_e_options.selection_strategy,
    "supported_subpoints": ", ".join(phase_e_result["supported_subpoint_ids"]) or "none",
    "unsupported_subpoints": ", ".join(phase_e_result["unsupported_subpoint_ids"]) or "none",
    "abstained": phase_e_result["abstained"],
})
print_table(phase_e_result["subpoint_support_rows"], columns=["subpoint_id", "label", "supported", "trusted_candidate_count", "trusted_doc_count", "top_candidate_title", "top_candidate_score"], max_rows=20, max_col_width=52)
print_section("Phase E - Lane Summary")
print_table(phase_e_result["lane_summary_rows"], columns=["lane", "candidate_count", "unique_docs", "top1_score", "top1_doc_id", "top1_view", "top1_title"], max_rows=20, max_col_width=54)
print_section("Phase E - Lane Overlap")
print_table(phase_e_result["lane_overlap_rows"], columns=["lane_left", "lane_right", "top_k", "overlap_count", "jaccard"], max_rows=20, max_col_width=32)
print_section("Phase E - Fused Candidate Preview")
print_table(phase_e_result["fused_preview_rows"], columns=["fused_rank", "doc_id", "title", "section_type", "pages", "fused_score", "selection_score", "trusted_subpoints", "lane_hits", "supporting_passages", "passage_only_support"], max_rows=20, max_col_width=52)
print_section("Phase E - QC")
print_table(phase_e_result["qc_rows"], columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)
